# 🤗 Getting Started with Transformers: Pipelines

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/01-Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
!pip install -q -U transformers
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['q_pipeline.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

The [Hugging Face Hub](https://huggingface.co/models) hosts more than two million models. The `transformers` library lets us use them with just a few lines of code.

`pipeline` is the simplest entry point: it bundles "preprocessing → model → postprocessing" into a single call.

In [ ]:
import torch
import transformers
from transformers import pipeline
from PIL import Image
from IPython.display import display
from urllib.request import urlopen
print(transformers.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"

## Sentiment Analysis

This model is multilingual, so we deliberately feed it Chinese reviews ("The beef noodles here are amazing, I'll definitely come back!" and "We waited an hour for our food, and the waiter was rude.") alongside English ones.

In [ ]:
classifier = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis", device=device)
classifier(["這家餐廳的牛肉麵超好吃，下次還要再來！",
            "等了一個小時才上菜，服務生態度又很差。",
            "The food was cold and the service was slow.",
            "I love this course!"])

## Zero-shot Classification

No training needed: just provide candidate labels. (Under the hood it uses Natural Language Inference: does "this text" entail "This is a news story about X"?)

The model is multilingual, so the labels and template can be in English while the headline is in Chinese ("TSMC announced today that it will build two more 2nm fabs in Kaohsiung"). The Chinese input is intentional.

In [ ]:
zero_shot = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device=device)
zero_shot("台積電今天宣布在高雄加蓋兩座 2 奈米晶圓廠",
          candidate_labels=["politics", "finance", "sports", "technology", "entertainment"],
          hypothesis_template="This is a news story about {}.")

## Named Entity Recognition (NER)

A Traditional Chinese model trained by CKIP Lab at Academia Sinica. The example sentence is intentionally Chinese ("Jensen Huang announced yesterday at Computex Taipei that NVIDIA will set up its Taiwan headquarters in the Beitou-Shilin Science Park.").

In [ ]:
ner = pipeline("token-classification", model="ckiplab/bert-base-chinese-ner",
               tokenizer="google-bert/bert-base-chinese", aggregation_strategy="simple", device=device)
entities = ner("黃仁勳昨天在台北國際電腦展宣布，NVIDIA 將在北投士林科技園區設立台灣總部。")
entities[:4]

CKIP's model uses BIOES tags (B = begin, I = inside, E = end, S = single), but the pipeline's aggregation only understands B/I, so some entities get split into pieces. We merge adjacent fragments of the same type ourselves:

In [ ]:
def merge_entities(ents):
    merged = []
    for e in ents:
        if merged and merged[-1]["entity_group"] == e["entity_group"] and merged[-1]["end"] == e["start"]:
            merged[-1]["word"] += e["word"]
            merged[-1]["end"] = e["end"]
        else:
            merged.append(dict(e))
    return [(e["word"].replace(" ", ""), e["entity_group"]) for e in merged]
merge_entities(entities)

## Fill-Mask

This is how "encoder" models like BERT are pre-trained: mask out a token in a sentence and let the model guess it.

We use the Chinese BERT on purpose here: the sentence means "The highest mountain in Taiwan is Yu[MASK]." and the answer is 山 ("mountain", as in Yushan).

In [ ]:
fill = pipeline("fill-mask", model="google-bert/bert-base-chinese", device=device)
fill(f"台灣最高的山是玉{fill.tokenizer.mask_token}。")

## Text Generation: a Small LLM

[Qwen3](https://huggingface.co/Qwen/Qwen3-0.6B) is Alibaba's open-source LLM family (Apache 2.0 license). The 0.6B version is tiny and runs fast on a T4.

If you give the pipeline a list of chat `messages`, it applies the chat template automatically.

In [ ]:
generator = pipeline("text-generation", model="Qwen/Qwen3-0.6B", dtype=torch.float16, device=device)
messages = [{"role": "user", "content": "Explain machine learning in one sentence. /no_think"}]
out = generator(messages, max_new_tokens=128)
answer = out[0]["generated_text"][-1]["content"]
print(answer)
print("Without the empty think block:", answer.split("</think>")[-1].strip())

`/no_think` is a Qwen3 instruction that turns off "thinking mode". We'll cover it in detail in the next unit.

## Automatic Speech Recognition

OpenAI Whisper supports 99 languages (including Chinese).

In [ ]:
import librosa
audio_url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"
with open("mlk.flac", "wb") as f:
    f.write(urlopen(audio_url).read())
audio, sr = librosa.load("mlk.flac", sr=16000)
from IPython.display import Audio
Audio(audio, rate=sr)

In [ ]:
asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo", dtype=torch.float16, device=device)
asr({"raw": audio, "sampling_rate": sr})

## Vision: Zero-shot Image Classification

[SigLIP 2](https://huggingface.co/google/siglip2-base-patch16-224) learns joint representations of images and text, so any text can serve as a class label.

In [ ]:
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
img = Image.open(urlopen(url)).convert("RGB")
display(img.resize((300, int(300 * img.height / img.width))))

In [ ]:
clip = pipeline("zero-shot-image-classification", model="google/siglip2-base-patch16-224", device=device)
clip(img, candidate_labels=["a cat", "a dog", "a lion", "a cat sitting on the grass", "a cat sitting on a sofa"])

## Vision: Depth Estimation

In [ ]:
depth = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=device)
result = depth(img)
display(result["depth"].resize((300, int(300 * img.height / img.width))))

## Q
Browse the [Hugging Face Hub](https://huggingface.co/models), pick a task that interests you (use the Tasks filter on the left), and try it with a pipeline. For example:
* `image-classification`
* `object-detection` (e.g. `PekingU/rtdetr_v2_r18vd`)
* `image-segmentation`
* `summarization`

In [ ]:
# %load q_pipeline.py
%run -i q_pipeline.py

## Freeing GPU Memory

A T4 has 15 GB of memory, so loading too many models will run out of room.

In [ ]:
import gc
del classifier, zero_shot, ner, fill, generator, asr, clip, depth
gc.collect()
torch.cuda.empty_cache()
print(f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")